# 数据级模型验证 · Validate model behavior by dataset ID

按 **ID** 抽取「图片 + Prompt」，并对比不同模型 / 检查点的输出 —— 从数据层面验证模型效果。

**示例数据集**：`peterant330/saliency-r1-8k`（本地：`.../datasets/saliency-r1-8k`）。
每条样本含：`question_id`(ID) · `problem`(问题) · `solution`(标准答案) · `image`(图片) · `bbox`(证据框，归一化) · `dataset`(子集)。

**三步流程**：① 指定一个 ID → ② 查看该 ID 的图片 + Prompt → ③ 查看 / 对比模型输出。

> 在 GPU 机器上用 OPD 的 uv 环境运行：`uv run jupyter lab`（或 `jupyter notebook`），打开本 notebook。
> 全程复用仓库现成代码（`baseline/*`、`vigos/*`），Prompt / 抽答案 / 判分与训练、`scripts/eval_opd.sh` 保持一致。
> 数据级查看（图片 / Prompt / 证据框）只需 `datasets`；模型推理（第 ⑤、⑥ 步）需要 GPU。

## ① 配置（通常只改这一格）

In [ ]:
# ====== 配置（按需修改）======
# 数据集：本地目录或 HuggingFace id。示例用 saliency-r1-8k（含证据框 bbox）。
DATASET = "/home/web_server/antispam/project/houshihao/datasets/saliency-r1-8k"
SPLIT   = "train"
SUBSETS = None          # 例如 ["docvqa", "textvqa"]；None = 全部子集

# 待对比的模型 / 检查点：名称 -> 路径（本地目录或 HF id）。可放多个一起对比。
M = "/home/web_server/antispam/project/houshihao/models"
MODELS = {
    "base": f"{M}/Qwen2.5-VL-3B-Instruct",
    # "opd":     "runs/opd_qwen25_3b_xxx/checkpoint-200",   # OPD 全参检查点直接指目录(无需 merge)
    # "teacher": f"{M}/Qwen2.5-VL-7B-Instruct",
}

# Prompt 风格（必须与训练一致）：think | freecot | reason | none
SYSTEM_PROMPT_STYLE = "think"
PROMPT_SUFFIX = ""

# 生成参数
DO_SAMPLE      = False  # False=贪心(可复现)；True=采样(看输出波动，可多跑几次)
TEMPERATURE    = 1.0
TOP_P          = 0.9
TOP_K          = 50
MAX_NEW_TOKENS = 1024

# 运行后端
DTYPE  = "bfloat16"
ATTN   = "sdpa"         # sdpa | flash_attention_2 | eager（第⑥步证据热力图需 eager）
DEVICE = "cuda"
print("配置完成：", len(MODELS), "个模型 ->", list(MODELS))

## ② 环境 & 复用仓库代码（imports、模型缓存）

In [ ]:
import os, sys
from collections import Counter

# 定位仓库根目录（含 baseline/ 与 vigos/），加入 sys.path 以便 import 复用。
def _find_repo_root(start):
    d = os.path.abspath(start)
    while d != os.path.dirname(d):
        if os.path.isdir(os.path.join(d, "baseline")) and os.path.isdir(os.path.join(d, "vigos")):
            return d
        d = os.path.dirname(d)
    return os.path.abspath(start)

REPO_ROOT = _find_repo_root(os.getcwd())
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("REPO_ROOT =", REPO_ROOT)

import torch
from PIL import Image, ImageDraw
from IPython.display import display

# 复用仓库：Prompt 构造 / 答案抽取 / 判分 / 通用样本字段抽取
from baseline.eval.opd_eval_prompt import build_general_eval_messages
from baseline.opd_data_collator import resolve_opd_system_prompt
from vigos.eval_utils import (
    extract_model_answer, extract_problem, extract_images, extract_ground_truth,
)
from vigos.answer_utils import grade_answer

SYSTEM_PROMPT = resolve_opd_system_prompt(SYSTEM_PROMPT_STYLE)
print("system prompt =", repr(SYSTEM_PROMPT))

In [ ]:
# ---- 模型 / 处理器：懒加载 + 缓存（重复运行不会重复加载）----
from transformers import AutoProcessor, AutoConfig

_PROC_CACHE, _MODEL_CACHE = {}, {}

def _resolve_vl_class(path):
    """解析 VL 模型类，兼容把 model_type 写成 *_text 的检查点（如 Saliency-R1）。"""
    cfg = AutoConfig.from_pretrained(path, trust_remote_code=True)
    mt = (getattr(cfg, "model_type", "") or "")
    base = mt[:-len("_text")] if mt.endswith("_text") else mt
    if base == "qwen2_5_vl":
        from transformers import Qwen2_5_VLForConditionalGeneration as C
        return C
    if base == "qwen3_vl":
        from transformers import Qwen3VLForConditionalGeneration as C
        return C
    from transformers import AutoModelForImageTextToText as C   # 通用兜底
    return C

def get_processor(path):
    if path not in _PROC_CACHE:
        _PROC_CACHE[path] = AutoProcessor.from_pretrained(
            path, trust_remote_code=True, use_fast=False)
    return _PROC_CACHE[path]

def get_model(name):
    path = MODELS[name]
    if name not in _MODEL_CACHE:
        print(f"[load] {name} <- {path}  (dtype={DTYPE}, attn={ATTN}) ...")
        cls = _resolve_vl_class(path)
        model = cls.from_pretrained(
            path, dtype=getattr(torch, DTYPE),
            attn_implementation=ATTN, trust_remote_code=True)
        _MODEL_CACHE[name] = model.to(DEVICE).eval()
        print(f"[load] {name} 就绪。")
    return _MODEL_CACHE[name]

def get_runner(name):
    return get_processor(MODELS[name]), get_model(name)

print("缓存就绪：get_processor / get_model / get_runner")

## ③ 加载数据集 + 建立 ID 索引

优先用 `saliency-r1-8k` 专用加载器（含证据框 `bbox` / 子集），仅返回**有合法 bbox** 的样本；
其它数据集自动回退到通用 OPD 加载器（`problem`/`image`/`answer`，ID 取 `question_id`/`problem_id`/`id`/行号，无 bbox）。

In [ ]:
def load_records(dataset, split, subsets):
    """返回 (records, has_bbox)。records 每条统一为
    {id, subset, problem, solution, image(PIL), bbox_norm(可None)}。"""
    # 1) 先试 saliency-r1-8k（含证据框 bbox）
    try:
        from baseline.probe.saliency_data import load_saliency_samples
        sams = load_saliency_samples(
            dataset, split, limit=None, subsets=subsets, max_bbox_area=None)
        if sams:
            recs = [dict(id=s.sample_id, subset=s.subset, problem=s.problem,
                         solution=s.solution, image=s.image, bbox_norm=s.bbox_norm)
                    for s in sams]
            return recs, True
    except Exception as e:
        print("[info] 非 saliency 数据集 / 加载器不适用，回退通用加载器：", repr(e))
    # 2) 通用回退（任意 OPD 数据集）
    from baseline.opd_dataset import load_opd_dataset
    ds = load_opd_dataset(dataset, split)
    recs = []
    for i in range(len(ds)):
        r = dict(ds[i])
        imgs = extract_images(r)
        rid = r.get("question_id", r.get("problem_id", r.get("id", i)))
        try:
            problem = extract_problem(r)
        except Exception:
            problem = str(r.get("problem") or r.get("question") or "")
        recs.append(dict(id=str(rid), subset=str(r.get("dataset", "") or ""),
                         problem=problem, solution=extract_ground_truth(r),
                         image=imgs[0] if imgs else None, bbox_norm=None))
    return recs, False

RECORDS, HAS_BBOX = load_records(DATASET, SPLIT, SUBSETS)
BY_ID = {r["id"]: r for r in RECORDS}     # 按 ID 取（question_id 一般唯一）
IDS   = list(BY_ID)
print(f"\n共 {len(RECORDS)} 条样本；含证据框 bbox：{HAS_BBOX}")
print("子集分布：", dict(Counter(r["subset"] for r in RECORDS)))
print("前 20 个可用 ID：", IDS[:20])
# 提示：也可按位置取样本 -> RECORDS[i]

## ④ 指定一个 ID → 查看图片 + Prompt

In [ ]:
def draw_bbox(image, bbox_norm, color=(255, 0, 0), width=4):
    """在图片副本上画出归一化证据框 (x1,y1,x2,y2)∈[0,1]（红框）。无框则原样返回。"""
    img = image.convert("RGB").copy()
    if bbox_norm:
        W, H = img.size
        x1, y1, x2, y2 = bbox_norm
        ImageDraw.Draw(img).rectangle([x1 * W, y1 * H, x2 * W, y2 * H],
                                      outline=color, width=width)
    return img

def build_prompt_text(processor, rec):
    """模型实际看到的完整 prompt（套用 chat template、加 generation prompt 后的字符串）。"""
    msgs = build_general_eval_messages(
        rec["problem"], [rec["image"]],
        system_prompt=SYSTEM_PROMPT, suffix=PROMPT_SUFFIX)
    return processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def show_sample(sample_id, processor=None):
    """显示某 ID 的图片(+证据框) / 问题 / 标准答案；传入 processor 时还打印完整 Prompt。"""
    rec = BY_ID[str(sample_id)]
    print("=" * 80)
    print(f"ID = {rec['id']}    子集 = {rec['subset']}")
    print(f"\n问题 problem:\n{rec['problem']}")
    print(f"\n标准答案 solution: {rec['solution']!r}")
    if rec.get("bbox_norm"):
        print(f"证据框 bbox(归一化 x1,y1,x2,y2): {tuple(round(v, 3) for v in rec['bbox_norm'])}")
    print(f"图片尺寸: {rec['image'].size}    （下图红框=证据区域，若有）")
    display(draw_bbox(rec["image"], rec.get("bbox_norm")))
    if processor is not None:
        print("\n---- 模型实际看到的完整 Prompt ----")
        print(build_prompt_text(processor, rec))
    return rec

In [ ]:
# 指定要检查的 ID（从上面打印的 IDS 里挑一个；也可直接填 question_id 数字/字符串）
SAMPLE_ID = IDS[0]

_first_proc = get_processor(MODELS[next(iter(MODELS))])   # 仅用其 chat template 展示 Prompt
rec = show_sample(SAMPLE_ID, processor=_first_proc)

## ⑤ 查看 / 对比模型输出

`generate_one` 用 HF transformers 交互推理；Prompt 与训练 / `eval_opd.sh` 一致。
默认贪心解码（可复现）；设 `DO_SAMPLE=True` 可观察采样波动。

In [ ]:
LBL = {True: "✅ 正确", False: "❌ 错误", None: "—（无标准答案）"}

def _gen_kwargs():
    kw = dict(max_new_tokens=MAX_NEW_TOKENS, do_sample=DO_SAMPLE)
    if DO_SAMPLE:
        kw.update(temperature=TEMPERATURE, top_p=TOP_P,
                  top_k=(TOP_K if TOP_K and TOP_K > 0 else None))
    return {k: v for k, v in kw.items() if v is not None}

@torch.no_grad()
def generate_one(name, sample_id):
    rec = BY_ID[str(sample_id)]
    proc, model = get_runner(name)
    msgs = build_general_eval_messages(
        rec["problem"], [rec["image"]],
        system_prompt=SYSTEM_PROMPT, suffix=PROMPT_SUFFIX)
    text = proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = proc(text=[text], images=[rec["image"]], return_tensors="pt").to(model.device)
    gen = model.generate(**inputs, **_gen_kwargs())
    completion = proc.batch_decode(
        gen[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]
    ans = extract_model_answer(completion)
    correct = grade_answer(ans, rec["solution"]) if rec["solution"] else None
    return dict(name=name, completion=completion, answer=ans, correct=correct)

def show_output(res, rec):
    print("=" * 80)
    print(f"[{res['name']}]  抽取答案 = {res['answer']!r}   "
          f"(标准答案 = {rec['solution']!r})   {LBL[res['correct']]}")
    print("-" * 80)
    print(res["completion"])

In [ ]:
# 单个模型跑指定 ID
res = generate_one(next(iter(MODELS)), SAMPLE_ID)
show_output(res, BY_ID[str(SAMPLE_ID)])

In [ ]:
# 同一 ID 对比所有 MODELS（先看图片，再逐个模型输出 + 汇总）
def compare_all(sample_id):
    rec = show_sample(sample_id)            # 图片 + 元信息（不打印 Prompt）
    rows = []
    for name in MODELS:
        rows.append(res := generate_one(name, sample_id))
        show_output(res, rec)
    print("\n" + "=" * 80 + "\n汇总：")
    print(f"{'模型':<14}{'抽取答案':<30}{'是否正确'}")
    for r in rows:
        print(f"{r['name']:<14}{str(r['answer'])[:28]:<30}{LBL[r['correct']]}")
    return rows

rows = compare_all(SAMPLE_ID)

## ⑥（可选 / 进阶）证据热力图：模型是否"看"向证据框？

这是本项目（evidence / saliency 对齐）的核心问题：模型回答某个 token 时，显著性是否落在 GT 证据框 `bbox` 内。
直接复用 `baseline/evidence/eval_transfer.py` 的机制，计算单样本的 **energy-in-bbox**（落框内的正显著性占比）与 **pointing**（显著性 argmax 是否命中框内），并把热力图叠加到原图。

注意：① 需要样本含 `bbox`；② 需 **eager** 注意力（会以 eager 另加载一份该模型，注意显存，必要时重启内核单独跑）；③ 模型输出需含 `\boxed{}` 才能定位答案 token。

In [ ]:
# （可选）单样本证据热力图。失败会打印原因并跳过，不影响上面的核心流程。
SALIENCY_MODEL = next(iter(MODELS))     # 选一个模型做热力图
try:
    import numpy as np
    from baseline.evidence.sanity_check import _load_model, _build_inputs
    from baseline.evidence.saliency_engine import resolve_model_parts
    from baseline.evidence.span_utils import parse_completion_spans
    from baseline.evidence.eval_transfer import (
        _saliency_map, _spans_to_positions, energy_and_pointing, resolve_layers,
    )

    rec = BY_ID[str(SAMPLE_ID)]
    assert rec.get("bbox_norm"), "该样本无 bbox，无法计算 energy-in-bbox（换一个含框的 ID）。"
    dtype = getattr(torch, DTYPE)
    proc = get_processor(MODELS[SALIENCY_MODEL])
    tok = getattr(proc, "tokenizer", proc)

    smodel = _load_model(MODELS[SALIENCY_MODEL], "eager", dtype)   # 单独 eager 加载
    parts = resolve_model_parts(smodel)
    layers = resolve_layers(parts, 4, None)                        # 末 4 层(与训练默认一致)

    inputs = _build_inputs(proc, rec["image"], rec["problem"])     # evidence 管线的输入构造
    prompt_len = int(inputs["input_ids"].shape[1])
    with torch.no_grad():
        gen = smodel.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    full_ids = gen[0]
    completion_ids = full_ids[prompt_len:]
    spans = parse_completion_spans(tok, completion_ids.tolist())
    print("---- completion ----\n", spans.text[:600])
    print(f"\nspans.valid={spans.valid}  reason={spans.reason}  answer={spans.answer}")
    if not spans.valid:
        raise RuntimeError(r"输出无可解析的 reason+\boxed{} 结构，无法定位答案 token；换个 ID 或让模型按格式输出。")

    pos = _spans_to_positions(spans, prompt_len, completion_ids, full_ids.device)
    smap = _saliency_map(smodel, parts, layers, full_ids, inputs, pos).float().cpu()  # [Hp,Wp]
    energy, pointing = energy_and_pointing(smap, rec["bbox_norm"])
    print(f"\n[{SALIENCY_MODEL}]  energy_in_bbox = {energy}   pointing(argmax 命中框内) = {pointing}")

    # 热力图叠加到原图（红色越亮=显著性越高），并画出 GT 框
    m = smap.clamp_min(0)
    m = (m / m.max()).numpy() if float(m.max()) > 0 else m.numpy()
    base = draw_bbox(rec["image"], rec["bbox_norm"]).convert("RGBA")
    W, H = base.size
    heat = Image.fromarray((m * 255).astype("uint8")).resize((W, H), Image.BILINEAR)
    overlay = Image.new("RGBA", (W, H), (255, 0, 0, 0))
    overlay.putalpha(heat.point(lambda a: int(a * 0.6)))
    display(Image.alpha_composite(base, overlay))
except Exception as e:
    import traceback; traceback.print_exc()
    print("\n[证据热力图] 跳过/失败：", repr(e))

## 附：小贴士

- **换数据集**：改 `DATASET`（本地目录或 HF id）。非 saliency 数据集自动走通用加载（无 bbox / 无第⑥步热力图）。
- **换模型 / 检查点**：往 `MODELS` 里加 `名称 -> 路径`。OPD 全参检查点直接指向 `runs/.../checkpoint-XXX` 目录（无需 merge）。
- **看输出波动**：设 `DO_SAMPLE=True`，多次运行第⑤步。
- **Prompt 风格必须与训练一致**：用 `SYSTEM_PROMPT_STYLE`（think / freecot / reason / none）。
- **复现 eval 指标**：本 notebook 用 HF transformers 交互推理，数值可能与 vLLM 评测略有差异；要对齐论文 / 榜单请用 `scripts/eval_opd.sh` / `scripts/eval_suite.sh`。
- **离线机器**：如需禁联网，单元格运行前 `os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"`。